In [ ]:
# CELL 1 — Connect Drive, packages, root
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import numpy as np, pandas as pd, json, hashlib, shutil, warnings, time
from collections import defaultdict
ROOT = Path('/content/drive/MyDrive/network-attack-forecasting')
V1 = ROOT / 'Synthetic_NAF_v1'
CAL = V1 / 'calibration'
V2 = ROOT / 'Synthetic_NAF_v2'
OUT = V2 / 'release_candidate'
OUT.mkdir(parents=True, exist_ok=True)
print('V1 calibration:', CAL)
print('V2 output:', OUT)

In [ ]:
# CELL 2 — Fixed model-facing contract; do not tune these from VAL/TEST model scores
FEATURES = [
 'Flow_Count','Flow_Duration_Mean','Flow_Duration_Std','Flow_Duration_Max',
 'Forward_Packets_Sum','Backward_Packets_Sum','Total_Packets',
 'Forward_Bytes_Sum','Backward_Bytes_Sum','Total_Bytes',
 'Packets_Per_Second','Bytes_Per_Second',
 'Forward_Backward_Packet_Ratio','Forward_Backward_Byte_Ratio'
]
PRIMITIVES = ['Flow_Count','Flow_Duration_Mean','Flow_Duration_Std',
              'Forward_Packets_Sum','Backward_Packets_Sum',
              'Forward_Bytes_Sum','Backward_Bytes_Sum']
EFFECT_MAP = {
 'Flow_Duration_Mean':'Flow_Duration_Seconds',
 'Forward_Packets_Sum':'Forward_Packets',
 'Backward_Packets_Sum':'Backward_Packets',
 'Forward_Bytes_Sum':'Forward_Bytes',
 'Backward_Bytes_Sum':'Backward_Bytes'
}
SPLIT_COUNTS = {'TRAIN':(210,350),'VALIDATION':(45,75),'TEST':(45,75)} # per-family attacks, benign
W=10; HORIZONS=[5,10,15]; MAX_H=15
SEED=20261022
EPISODE_MINUTES=110; ONSET_MINUTE=65; ATTACK_MINUTES=25
MAX_ATTEMPTS=30
# Fixed, simple, data-only quality gates (assumptions, not trained cutoffs).
QC_MIN_LATE_VS_EARLY=0.04
QC_MIN_ATTACK_VS_PRE=0.03
QC_MAX_PRE_TO_ATTACK=1.5
STRICT_RELEASE=True

In [ ]:
# CELL 3 — Load authoritative source-conditioned files, fail loudly if missing
FILES = {
 'blocks':CAL/'11_2_safe_benign_block_rows.csv',
 'effects':CAL/'11_1_source_family_log_effect_vectors.csv',
 'policy':CAL/'11_2_family_generation_policy.csv',
 'profile':CAL/'11_1_flow_profile_quantiles.csv',
 'minute':CAL/'11_1_real_minute_feature_quantiles.csv',
 'intensity':CAL/'11_1_family_attack_flow_intensity_quantiles.csv'
}
missing=[str(p) for p in FILES.values() if not p.is_file()]
if missing: raise FileNotFoundError('Required v1 calibration artifact(s) missing: '+repr(missing))
def digest(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
 return h.hexdigest()
blocks=pd.read_csv(FILES['blocks'],low_memory=False)
effects=pd.read_csv(FILES['effects'],low_memory=False)
policy=pd.read_csv(FILES['policy'],low_memory=False)
profile=pd.read_csv(FILES['profile'],low_memory=False)
minuteq=pd.read_csv(FILES['minute'],low_memory=False)
intensity=pd.read_csv(FILES['intensity'],low_memory=False)
need_blocks=['Calibration_Source','Calibration_Pool','Calibration_Group_ID',
             'Safe_Benign_Block_ID','Block_Row_Position',*FEATURES]
need_effects=['Calibration_Source','Synthetic_Family','Feature','Effect_Log1p']
for name,df,cols in [('blocks',blocks,need_blocks),('effects',effects,need_effects)]:
 absent=sorted(set(cols)-set(df.columns))
 if absent:raise ValueError(f'{name} missing fields: {absent}')
for f in FEATURES:blocks[f]=pd.to_numeric(blocks[f],errors='raise')
assert np.isfinite(blocks[FEATURES].to_numpy(float)).all()
assert (blocks[FEATURES]>=0).all().all()
assert (blocks.groupby('Safe_Benign_Block_ID').size()==5).all()
effects['Effect_Log1p']=pd.to_numeric(effects['Effect_Log1p'],errors='raise')
FAMILIES=sorted(effects['Synthetic_Family'].dropna().astype(str).unique())
assert len(FAMILIES)==5, f'Expected exactly five families: {FAMILIES}'
print('FAMILIES:',FAMILIES)
print('Source pools:', blocks.groupby(['Calibration_Source','Calibration_Pool']).Calibration_Group_ID.nunique())
print('Frozen v1 predictors:',len(FEATURES))

In [ ]:
# CELL 4 — Freeze source-group ownership by split and determine supported source/family pairs
# Each real calibration group belongs to exactly ONE partition: no group can appear across splits.
# Existing CAL_TRAIN/CAL_VALIDATION/CAL_TEST mapping is authoritative.
POOL_MAP={'TRAIN':'CAL_TRAIN','VALIDATION':'CAL_VALIDATION','TEST':'CAL_TEST'}
POOL_ALIAS={'TRAIN':'TRAIN','VAL':'VALIDATION','VALIDATION':'VALIDATION','TEST':'TEST',
            'CAL_TRAIN':'TRAIN','CAL_VALIDATION':'VALIDATION','CAL_VAL':'VALIDATION','CAL_TEST':'TEST'}
blocks['Split']=blocks['Calibration_Pool'].astype(str).str.upper().map(POOL_ALIAS)
if blocks.Split.isna().any():raise ValueError('Unrecognized calibration pools: '+str(blocks.loc[blocks.Split.isna(),'Calibration_Pool'].unique()))
source_split_groups=blocks[['Calibration_Source','Calibration_Group_ID','Split']].drop_duplicates()
assert source_split_groups.groupby(['Calibration_Source','Calibration_Group_ID']).Split.nunique().max()==1
block_lookup={}; groups_by_source_split=defaultdict(list)
for (source,split,gid,bid),x in blocks.groupby(['Calibration_Source','Split','Calibration_Group_ID','Safe_Benign_Block_ID'],sort=False):
 block_lookup[(source,split,str(gid),str(bid))]=x.sort_values('Block_Row_Position')[FEATURES].to_numpy(float)
for (source,split,gid),df in blocks.groupby(['Calibration_Source','Split','Calibration_Group_ID'],sort=False):
 bids=sorted(df['Safe_Benign_Block_ID'].astype(str).unique())
 groups_by_source_split[(source,split)].append((str(gid),bids))
effect_lookup={(str(r.Calibration_Source),str(r.Synthetic_Family),str(r.Feature)):float(r.Effect_Log1p)
               for r in effects.itertuples(index=False)}
SOURCES=sorted(set(blocks.Calibration_Source.astype(str)))
SUPPORTED={family:[source for source in SOURCES if all((source,family,f) in effect_lookup for f in EFFECT_MAP.values())
                                           and all((source,sp) in groups_by_source_split for sp in SPLIT_COUNTS)]
           for family in FAMILIES}
if any(not x for x in SUPPORTED.values()):raise ValueError(f'No valid calibration source across every split for family: {SUPPORTED}')
# Restrict to already-supported source/family combinations; NEVER CIC baseline + UNSW effect.
print('Supported source/family pairs:',SUPPORTED)
print('Unique group counts:',{str(k):len(v) for k,v in groups_by_source_split.items()})

In [ ]:
# CELL 5 — Recompute derived columns from simulated primitives; do not manipulate ratios independently
def reconstruct(p):
 out=np.zeros((len(p),len(FEATURES)),np.float64)
 d=dict(zip(PRIMITIVES,p.T))
 flow=np.maximum(d['Flow_Count'],1.)
 duration=np.maximum(d['Flow_Duration_Mean'],0.)
 std=np.maximum(d['Flow_Duration_Std'],0.)
 out[:,FEATURES.index('Flow_Count')]=flow
 out[:,FEATURES.index('Flow_Duration_Mean')]=duration
 out[:,FEATURES.index('Flow_Duration_Std')]=std
 out[:,FEATURES.index('Flow_Duration_Max')]=duration+1.5*std
 for c in ['Forward_Packets_Sum','Backward_Packets_Sum','Forward_Bytes_Sum','Backward_Bytes_Sum']:
  out[:,FEATURES.index(c)]=np.maximum(d[c],0.)
 fpk=out[:,FEATURES.index('Forward_Packets_Sum')]
 bpk=out[:,FEATURES.index('Backward_Packets_Sum')]
 fby=out[:,FEATURES.index('Forward_Bytes_Sum')]
 bby=out[:,FEATURES.index('Backward_Bytes_Sum')]
 out[:,FEATURES.index('Total_Packets')]=fpk+bpk
 out[:,FEATURES.index('Total_Bytes')]=fby+bby
 out[:,FEATURES.index('Packets_Per_Second')]=(fpk+bpk)/60.
 out[:,FEATURES.index('Bytes_Per_Second')]=(fby+bby)/60.
 out[:,FEATURES.index('Forward_Backward_Packet_Ratio')]=fpk/np.maximum(bpk,1.)
 out[:,FEATURES.index('Forward_Backward_Byte_Ratio')]=fby/np.maximum(bby,1.)
 return out

def robust_signal(x,baseline):
 # Data-only relative deviation in independent primitive log1p space.
 # Scores ARE QC metadata, never supplied to a forecasting model.
 j=[FEATURES.index(k) for k in PRIMITIVES]
 lo=np.log1p(np.maximum(x[:,j],0))
 ref=np.log1p(np.maximum(baseline[:,j],0))
 scale=np.maximum(np.median(np.abs(ref-np.median(ref,axis=0)),axis=0)*1.4826,0.10)
 return np.median(np.abs(lo-ref)/scale,axis=1)

def source_effects(source,family):
 vals=np.array([effect_lookup[(source,family,EFFECT_MAP[k])] for k in list(EFFECT_MAP)])
 if not np.isfinite(vals).all(): raise ValueError(f'Invalid family effects {source},{family}')
 return vals

In [ ]:
# CELL 6 — Episode generator: shared temporal baseline + smooth pre-onset family trajectory
# Simulated pre-attack signal is an explicitly controlled assumption. No actual CIC/UNSW
# pre-onset history is claimed here. Anti-shortcut: benign anomalies, per-episode variation.

def make_baseline(source,split,rng):
 gid,bids=groups_by_source_split[(source,split)][int(rng.integers(len(groups_by_source_split[(source,split)])))]
 block=block_lookup[(source,split,gid,bids[int(rng.integers(len(bids)))])]
 # 5-min real calibration trajectories recycled with slow AR correlated multiplicative variation.
 # Distinct episodes are NOT independent empirical observations; see release limitations.
 indices=np.arange(EPISODE_MINUTES)%5
 prim=np.column_stack([block[indices,FEATURES.index(k)] for k in PRIMITIVES])
 noise=rng.normal(0,0.025,(EPISODE_MINUTES,len(PRIMITIVES)))
 for t in range(1,EPISODE_MINUTES):noise[t]=0.88*noise[t-1]+noise[t]
 prim=np.maximum(prim*np.exp(noise),0.)
 prim[:,0]=np.maximum(1.,np.rint(prim[:,0]))
 return prim,gid,bids

def generate_episode(split,family,seed,hard=False,source=None,precursor_len=None):
 rng=np.random.default_rng(int(seed))
 source=source or str(rng.choice(SUPPORTED[family] if family is not None else
                       [s for s in SOURCES if (s,split) in groups_by_source_split]))
 base,gid,bids=make_baseline(source,split,rng)
 p=base.copy(); onset=ONSET_MINUTE if family is not None else -1
 precursor_len=int(precursor_len or rng.integers(15,31)) if family is not None else 0
 if family is not None:
  start=onset-precursor_len
  effect=source_effects(source,family)
  # Log-space effect for five source-calibrated flow primitives.
  # Aggregate packets/bytes are reconstructed from per-flow effect*Flow_Count.
  # No model outcomes participate in amplitude sampling or acceptance.
  strength=float(rng.uniform(0.35,0.85))
  magnitude=float(np.clip(np.median(np.abs(effect)),0.10,1.8))
  # A bounded, developing precursor (< full attack), plus attack phase.
  progress=np.arange(1,precursor_len+1)/precursor_len
  alpha=0.08+0.59*(3*progress**2-2*progress**3)
  alpha=np.minimum(alpha*float(rng.uniform(.85,1.10)),.75)
  shape=np.zeros(EPISODE_MINUTES)
  shape[start:onset]=alpha
  shape[onset:onset+ATTACK_MINUTES]=float(rng.uniform(.90,1.15))
  # Simultaneously change flow count and calibrated per-flow behaviours; all bounded.
  # Flow count increase is a simulation assumption; not fabricated as a sourced family effect.
  flowlog=shape*float(rng.uniform(.15,.50))*min(1.0,magnitude)
  p[:,0]=np.maximum(1,np.rint(base[:,0]*np.exp(flowlog)))
  for k_idx,feature in enumerate(EFFECT_MAP):
   j=PRIMITIVES.index(feature)
   log_effect=effect[k_idx]*strength*shape
   if feature.endswith('_Sum'):
    base_pf=np.maximum(base[:,j],0)/np.maximum(base[:,0],1)
    p[:,j]=np.maximum(0,np.expm1(np.log1p(base_pf)+log_effect))*p[:,0]
   else:p[:,j]=np.maximum(0,np.expm1(np.log1p(base[:,j])+log_effect))
 else:
  # Substantial benign transient, WITHOUT onset, to challenge anomaly-only detectors.
  if hard:
   start=int(rng.integers(25,50)); length=int(rng.integers(8,22))
   amplitude=float(rng.uniform(.25,.90))
   curve=np.sin(np.linspace(0,np.pi,length))**2
   affected=rng.choice([0,3,4,5,6],size=int(rng.integers(1,4)),replace=False)
   for j in affected:p[start:start+length,j]*=np.exp(amplitude*curve*rng.choice([.75,1]))
 p[:,0]=np.maximum(1.,np.rint(p[:,0]))
 features=reconstruct(p)
 return {'features':features,'base':reconstruct(base),'onset':onset,
         'source':source,'group_id':gid,'block_ids':';'.join(map(str,bids)),
         'precursor_len':precursor_len,'hard':bool(hard)}

def gate(ep,family):
 x=ep['features']; base=ep['base']
 if not np.isfinite(x).all() or (x<0).any():return False,'NONFINITE_OR_NEGATIVE',{}
 if (x[:,FEATURES.index('Flow_Duration_Max')]<x[:,FEATURES.index('Flow_Duration_Mean')]).any():return False,'DURATION_INVARIANT',{}
 if family is None:return True,'PASS',{}
 onset=ep['onset']; start=onset-ep['precursor_len']
 dev=robust_signal(x,base)
 length=ep['precursor_len']; mid=start+length//2
 early=float(np.mean(dev[start:mid]));late=float(np.mean(dev[mid:onset]))
 attack=float(np.mean(dev[onset:onset+ATTACK_MINUTES]))
 metrics={'signal_early':early,'signal_late':late,'signal_attack':attack,
          'late_minus_early':late-early,'attack_minus_late':attack-late}
 if late-early<QC_MIN_LATE_VS_EARLY:return False,'NOT_DEVELOPING',metrics
 if attack-late<QC_MIN_ATTACK_VS_PRE:return False,'NO_ATTACK_TRANSITION',metrics
 if late>QC_MAX_PRE_TO_ATTACK*max(attack,1e-6):return False,'PRECURSOR_TOO_STRONG',metrics
 return True,'PASS',metrics

In [ ]:
# CELL 7 — Predeclare episode manifest; split at the *real calibration group* level
# Counts are fixed; generation may REJECT an episode but will NOT relax QC thresholds.
rng=np.random.default_rng(SEED)
manifest=[]
for split,(per_family,benign) in SPLIT_COUNTS.items():
 for family in FAMILIES:
  sources=SUPPORTED[family]
  for i in range(per_family):
   manifest.append({'Episode_ID':f'{split}_A_{family}_{i:04d}','Split':split,
                    'Family':family,'Hard_Benign':False,'Source':sources[i%len(sources)],
                    'Seed':int(rng.integers(1,2**62))})
 for i in range(benign):
  choices=[s for s in SOURCES if (s,split) in groups_by_source_split]
  if not choices:raise ValueError(f'No benign calibration source in {split}')
  manifest.append({'Episode_ID':f'{split}_B_{i:04d}','Split':split,
                   'Family':None,'Hard_Benign':i<(benign//2 + (1 if split=='TEST' else 0)),'Source':choices[i%len(choices)],
                   'Seed':int(rng.integers(1,2**62))})
manifest=pd.DataFrame(manifest)
assert len(manifest)==2000
assert manifest.groupby('Split').size().to_dict()=={'TEST':300,'TRAIN':1400,'VALIDATION':300}
assert manifest.Episode_ID.is_unique and manifest.Seed.is_unique
print(manifest.groupby(['Split','Family'],dropna=False).size())

In [ ]:
# CELL 8 — Generate, measure, fail closed if QC cannot be met
accepted=[]; failures=[]; episode_rows=[]
for i,row in enumerate(manifest.itertuples(index=False)):
 ep=None;passed=False;reason='NO_ATTEMPTS';metrics={}
 for attempt in range(MAX_ATTEMPTS):
  newseed=int(row.Seed)+100003*attempt
  candidate=generate_episode(row.Split,row.Family,newseed,bool(row.Hard_Benign),source=row.Source)
  good,reason,metrics=gate(candidate,row.Family)
  if good:ep=candidate;passed=True;break
  failures.append({'Episode_ID':row.Episode_ID,'Attempt':attempt,'Failure':reason})
 if not passed:
  failpath=OUT/'generation_failures.csv'
  pd.DataFrame(failures).to_csv(failpath,index=False)
  raise RuntimeError(f'QC FAILURE episode={row.Episode_ID}, after {MAX_ATTEMPTS} attempts; last reason={reason}. '
                     f'No incomplete dataset was released. See {failpath}')
 x=ep['features'].astype(np.float32)
 df=pd.DataFrame(x,columns=FEATURES)
 df.insert(0,'Minute',np.arange(len(df),dtype=np.int16))
 df.insert(0,'Episode_ID',row.Episode_ID)
 episode_rows.append(df)
 accepted.append({'Episode_ID':row.Episode_ID,'Split':row.Split,'Family':row.Family or 'BENIGN',
                  'Hard_Benign':bool(row.Hard_Benign),'Source':ep['source'],
                  'Calibration_Group_ID':ep['group_id'], 'Calibration_Block_IDs':ep['block_ids'],
                  'Onset_Minute':ep['onset'],'Precursor_Length':ep['precursor_len'],
                  'Seed':row.Seed,'Attempts':attempt+1,**metrics})
 if (i+1)%200==0:print('ACCEPTED',i+1,'/',len(manifest))
meta=pd.DataFrame(accepted); minutes=pd.concat(episode_rows,ignore_index=True)
assert len(meta)==2000 and len(minutes)==2000*EPISODE_MINUTES
assert not minutes.duplicated(['Episode_ID','Minute']).any()
print('Generation completed; rejected proposals:',len(failures))

In [ ]:
# CELL 9 — Causal pre-onset labels, explicit right-censoring, and model-ready W10 sequences
# t is the LAST observed minute. Strict positive iff onset in (t,t+H].
# At the end of a benign episode, horizon labels are CENSORED, not negative.
# At/post attack onset, prediction rows are excluded from training.
# Regression ETA valid only when onset is observed within max horizon.
WINDOW_KEYS=['Episode_ID','Minute']
for split in SPLIT_COUNTS:
 keys=[]; X=[]; y=[]; tto=[]; fam=[]; horizon_y=[]; horizon_mask=[]
 for entry in meta.loc[meta.Split==split].itertuples(index=False):
  z=minutes.loc[minutes.Episode_ID==entry.Episode_ID,FEATURES].to_numpy(np.float32)
  for t in range(W-1,EPISODE_MINUTES):
   if entry.Onset_Minute>=0 and t>=entry.Onset_Minute:break
   future=int(entry.Onset_Minute-t) if entry.Onset_Minute>=0 else -1
   # For benign, all 15 future minutes must exist for a valid 'no event in 15min'.
   complete=(t+MAX_H<EPISODE_MINUTES)
   if not complete:continue
   keys.append((entry.Episode_ID,t));X.append(z[t-W+1:t+1])
   y.append(future if 1<=future<=MAX_H else 0)
   tto.append(future if future>0 else -1)
   fam.append(FAMILIES.index(entry.Family)+1 if 1<=future<=MAX_H else 0)
   horizon_y.append([int(1<=future<=h) for h in HORIZONS]);horizon_mask.append([1]*len(HORIZONS))
 if not X:raise RuntimeError('No model samples for '+split)
 a=np.stack(X); yy=np.asarray(y,dtype=np.int16); ff=np.asarray(fam,dtype=np.int8)
 # y=0 means 'no onset in next 15m'; y=k means onset k minutes later.
 assert np.isfinite(a).all() and np.all((yy>=0)&(yy<=15))
 # Split files use original numerical units. Fit scalers ONLY on TRAIN when modeling.
 np.savez_compressed(OUT/f'windows_W10_{split}.npz',X=a,onset_class=yy,family_class=ff,
                     y_horizons=np.asarray(horizon_y,dtype=np.int8),
                     horizon_mask=np.asarray(horizon_mask,dtype=np.int8),
                     episode_id=np.array([v[0] for v in keys]),minute=np.array([v[1] for v in keys]))
 print(split,a.shape,'H5/H10/H15 prevalence:',np.mean(horizon_y,axis=0).round(4))

In [ ]:
# CELL 10 — Full data-only release verification and hash manifest
assert meta.Episode_ID.is_unique
assert len(meta[meta.Family!='BENIGN'])==1500
assert len(meta[meta.Family=='BENIGN'])==500
assert meta[meta.Family=='BENIGN'].Hard_Benign.sum()==250
assert all(meta.loc[meta.Family==f].shape[0]==300 for f in FAMILIES)
assert all(meta.loc[meta.Family==f,'Precursor_Length'].between(15,30).all() for f in FAMILIES)
assert (meta.loc[meta.Family!='BENIGN','signal_late'] > meta.loc[meta.Family!='BENIGN','signal_early']).all()
assert (meta.loc[meta.Family!='BENIGN','signal_attack'] > meta.loc[meta.Family!='BENIGN','signal_late']).all()
# CALIBRATION GROUP ISOLATION: each source/group may only occur in one model partition.
if meta.groupby(['Source','Calibration_Group_ID']).Split.nunique().max()!=1:
 raise RuntimeError('Real calibration group crosses synthetic TRAIN/VAL/TEST')
assert all(minutes.groupby('Episode_ID').Minute.apply(lambda z:np.array_equal(z.to_numpy(),np.arange(EPISODE_MINUTES))))
num=minutes[FEATURES].to_numpy(float)
assert np.isfinite(num).all() and (num>=0).all()
assert np.allclose(minutes.Total_Packets,minutes.Forward_Packets_Sum+minutes.Backward_Packets_Sum,rtol=1e-5,atol=1e-3)
assert np.allclose(minutes.Total_Bytes,minutes.Forward_Bytes_Sum+minutes.Backward_Bytes_Sum,rtol=1e-5,atol=1e-2)
# Persist separate predictor/metadata artifacts. Metadata is NEVER a predictor.
minutes.to_parquet(OUT/'minute_predictors.parquet',index=False)
meta.to_csv(OUT/'episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv',index=False)
pd.DataFrame(failures).to_csv(OUT/'qc_rejections.csv',index=False)
contract={'dataset':'Synthetic_NAF_v2','status':'SYNTHETIC_RELEASE_CANDIDATE',
 'features':FEATURES,'families':FAMILIES,'episode_count':2000,
 'train_val_test_counts':{'TRAIN':1400,'VALIDATION':300,'TEST':300},
 'history_minutes':W,'onset_horizon_minutes':HORIZONS,'minute_cadence':1,
 'synthetic_precursor_length_range':[15,30],
 'source_calibration':'v1 11.1/11.2 source-conditioned artifacts',
 'empirical_precursor_claim':False,'generator_seed':SEED,
 'generator_assumptions':'Source-calibrated effects with hand-simulated pre-onset timing/intensity; no real precursor evidence',
 'independence_limitation':'5-minute empirical calibration blocks are reused WITHIN splits; 2000 synthetic episodes are not 2000 independent real traffic captures',
 'QC_THRESHOLDS':{'late_vs_early':QC_MIN_LATE_VS_EARLY,'attack_vs_pre':QC_MIN_ATTACK_VS_PRE,'max_pre_to_attack':QC_MAX_PRE_TO_ATTACK},
 'inputs_sha256':{k:digest(p) for k,p in FILES.items()}}
(OUT/'FEATURE_AND_DATASET_CONTRACT.json').write_text(json.dumps(contract,indent=2))
(OUT/'QC_SUMMARY.json').write_text(json.dumps({
  'status':'DATA_ONLY_GATES_PASSED', 'accepted_episodes':len(meta),
  'rejected_proposals':len(failures),'episode_splits':meta.groupby('Split').size().to_dict()},indent=2))
files_to_hash=sorted(p for p in OUT.iterdir() if p.is_file() and p.name!='SHA256SUMS.json')
(OUT/'SHA256SUMS.json').write_text(json.dumps({p.name:digest(p) for p in files_to_hash},indent=2))
print('DATA-ONLY RELEASE GATES: PASS')
print('Output folder:',OUT)
print('Files:',[p.name for p in OUT.iterdir()])

In [ ]:
# =====================================================================================
# SYNTHETIC_NAF_V2 — CORRECTED DATASET BUILDER
#
# Fixes:
#   1. Randomized attack onset (previously fixed at minute 65)
#   2. Removes deterministic 5-minute baseline repetition
#   3. Hard benign events overlap attack-development periods
#   4. Preserves source-specific calibration and split isolation
#   5. Regenerates all 2,000 episodes with automatic QC
#   6. Rebuilds causal W10 / H5 / H10 / H15 forecasting windows
#   7. Performs TRAIN-only shortcut diagnostics
#   8. Saves a separate, hashed release candidate
#
# PREREQUISITE:
#   Run in the SAME Colab runtime where original V2 cells have already run.
#
# DOES NOT:
#   Modify Dataset_Audit.ipynb
#   Overwrite the original V2 release_candidate
#   Train forecasting models
#   Evaluate held-out TEST model performance
#
# =====================================================================================

import numpy as np
import pandas as pd
import json
import hashlib

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score, roc_auc_score


# =====================================================================================
# 1 — PATHS AND CONFIGURATION
# =====================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

OUT_CORRECTED = (
    ROOT
    / "Synthetic_NAF_v2"
    / "release_candidate_corrected"
)

if OUT_CORRECTED.exists() and any(OUT_CORRECTED.iterdir()):
    raise RuntimeError(
        f"Output folder already contains files: {OUT_CORRECTED}. "
        "Move or rename it before another run to avoid mixing releases."
    )

OUT_CORRECTED.mkdir(parents=True, exist_ok=True)

EPISODE_MINUTES = 110
ONSET_LOW = 46
ONSET_HIGH = 82
ATTACK_MINUTES = 25

BASELINE_SEGMENT_MIN = 3
BASELINE_SEGMENT_MAX = 12

MAX_ATTEMPTS = 30

QC_MIN_LATE_VS_EARLY = 0.04
QC_MIN_ATTACK_VS_PRE = 0.03
QC_MAX_PRE_TO_ATTACK = 1.5

W = 10
HORIZONS = [5, 10, 15]
MAX_H = 15

print("=" * 90)
print("SYNTHETIC_NAF_V2 — CORRECTED DATASET GENERATION")
print("=" * 90)
print("Output:", OUT_CORRECTED)


# =====================================================================================
# 2 — VERIFY EXISTING NOTEBOOK VARIABLES
# =====================================================================================

required_names = [
    "FEATURES",
    "PRIMITIVES",
    "FAMILIES",
    "SPLIT_COUNTS",
    "manifest",
    "block_lookup",
    "groups_by_source_split",
    "reconstruct",
    "robust_signal",
    "source_effects",
    "EFFECT_MAP",
    "gate",
]

missing_names = [
    name for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Original V2 notebook variables are missing. "
        "Run its setup/manifest cells first. Missing: "
        + str(missing_names)
    )

assert len(FEATURES) == 14
assert len(FAMILIES) == 5
assert len(manifest) == 2000
assert manifest.Episode_ID.is_unique
assert manifest.Seed.is_unique

print("Original calibration and manifest: LOADED")
print("Features:", len(FEATURES))
print("Families:", FAMILIES)
print("Episodes:", len(manifest))


# =====================================================================================
# 3 — CORRECTED NON-PERIODIC BASELINE
# =====================================================================================

def make_baseline_v2(source, split, rng):

    available = groups_by_source_split[(source, split)]

    if not available:
        raise RuntimeError(
            f"No calibration group: {source}, {split}"
        )

    group_id, block_ids = available[
        int(rng.integers(len(available)))
    ]

    # All anchors come from one split-isolated calibration group.
    library = np.concatenate(
        [
            block_lookup[
                (source, split, group_id, block_id)
            ]
            for block_id in block_ids
        ],
        axis=0
    )

    observed = np.column_stack(
        [
            library[:, FEATURES.index(feature)]
            for feature in PRIMITIVES
        ]
    )

    transformed = np.log1p(
        np.maximum(observed, 0)
    )

    output = np.empty(
        (EPISODE_MINUTES, len(PRIMITIVES)),
        dtype=np.float64
    )

    cursor = 0

    while cursor < EPISODE_MINUTES:

        segment_length = int(
            rng.integers(
                BASELINE_SEGMENT_MIN,
                BASELINE_SEGMENT_MAX + 1
            )
        )

        count = min(
            segment_length,
            EPISODE_MINUTES - cursor
        )

        anchor_a = int(
            rng.integers(len(transformed))
        )

        anchor_b = int(
            rng.integers(len(transformed))
        )

        progress = np.linspace(
            0, 1, count + 2
        )[1:-1]

        progress = (
            progress**2
            * (3 - 2 * progress)
        )

        output[cursor:cursor + count] = (
            (1 - progress[:, None])
            * transformed[anchor_a]
            +
            progress[:, None]
            * transformed[anchor_b]
        )

        cursor += count

    # Temporally correlated variation
    noise = rng.normal(
        0,
        0.035,
        output.shape
    )

    for t in range(1, len(noise)):
        noise[t] = (
            0.78 * noise[t - 1]
            + noise[t]
        )

    output += noise

    # Smooth interpolation boundaries
    padded = np.pad(
        output,
        ((2, 2), (0, 0)),
        mode="edge"
    )

    output = np.array(
        [
            padded[t:t + 5].mean(axis=0)
            for t in range(EPISODE_MINUTES)
        ]
    )

    primitive_values = np.maximum(
        np.expm1(output),
        0
    )

    primitive_values[:, 0] = np.maximum(
        1,
        np.rint(primitive_values[:, 0])
    )

    return primitive_values, group_id, block_ids


# =====================================================================================
# 4 — CORRECTED ATTACK AND BENIGN EPISODE GENERATION
# =====================================================================================

def generate_episode_v2(
    split,
    family,
    seed,
    hard=False,
    source=None
):

    rng = np.random.default_rng(int(seed))

    baseline, group_id, block_ids = make_baseline_v2(
        source,
        split,
        rng
    )

    primitives = baseline.copy()

    onset = -1
    precursor_len = 0
    benign_anomaly_start = -1

    if family is not None:

        onset = int(
            rng.integers(
                ONSET_LOW,
                ONSET_HIGH + 1
            )
        )

        precursor_len = int(
            rng.integers(15, 31)
        )

        start = onset - precursor_len

        effects = source_effects(
            source,
            family
        )

        strength = float(
            rng.uniform(0.35, 0.85)
        )

        magnitude = float(
            np.clip(
                np.median(np.abs(effects)),
                0.10,
                1.8
            )
        )

        # Smooth developing precursor
        progress = (
            np.arange(1, precursor_len + 1)
            / precursor_len
        )

        alpha = (
            0.08
            +
            0.59
            *
            (
                3 * progress**2
                - 2 * progress**3
            )
        )

        alpha *= float(
            rng.uniform(0.85, 1.10)
        )

        alpha = np.minimum(alpha, 0.75)

        shape = np.zeros(
            EPISODE_MINUTES,
            dtype=float
        )

        shape[start:onset] = alpha

        shape[
            onset:onset + ATTACK_MINUTES
        ] = float(
            rng.uniform(0.90, 1.15)
        )

        # Flow-count evolution
        flow_effect = (
            shape
            *
            float(rng.uniform(0.15, 0.50))
            *
            min(1.0, magnitude)
        )

        primitives[:, 0] = np.maximum(
            1,
            np.rint(
                baseline[:, 0]
                * np.exp(flow_effect)
            )
        )

        # Source-conditioned family effects
        for effect_index, feature in enumerate(EFFECT_MAP):

            primitive_index = PRIMITIVES.index(feature)

            log_effect = (
                effects[effect_index]
                * strength
                * shape
            )

            if feature.endswith("_Sum"):

                baseline_per_flow = (
                    baseline[:, primitive_index]
                    /
                    np.maximum(baseline[:, 0], 1)
                )

                adjusted = np.maximum(
                    0,
                    np.expm1(
                        np.log1p(baseline_per_flow)
                        + log_effect
                    )
                )

                primitives[:, primitive_index] = (
                    adjusted * primitives[:, 0]
                )

            else:

                primitives[:, primitive_index] = np.maximum(
                    0,
                    np.expm1(
                        np.log1p(
                            baseline[:, primitive_index]
                        )
                        + log_effect
                    )
                )

    elif hard:

        # Benign anomaly timing overlaps attack development.
        benign_anomaly_start = int(
            rng.integers(
                ONSET_LOW - 30,
                ONSET_HIGH + 1
            )
        )

        duration = int(
            rng.integers(8, 31)
        )

        amplitude = float(
            rng.uniform(0.25, 0.90)
        )

        curve = (
            np.sin(
                np.linspace(0, np.pi, duration)
            )**2
        )

        affected = rng.choice(
            [0, 3, 4, 5, 6],
            size=int(rng.integers(1, 4)),
            replace=False
        )

        for feature_index in affected:

            primitives[
                benign_anomaly_start:
                benign_anomaly_start + duration,
                feature_index
            ] *= np.exp(
                amplitude
                * curve
                * float(rng.choice([0.75, 1.0]))
            )

    primitives[:, 0] = np.maximum(
        1,
        np.rint(primitives[:, 0])
    )

    return {
        "features": reconstruct(primitives),
        "base": reconstruct(baseline),
        "onset": onset,
        "source": source,
        "group_id": group_id,
        "block_ids": ";".join(map(str, block_ids)),
        "precursor_len": precursor_len,
        "hard": bool(hard),
        "benign_anomaly_start": benign_anomaly_start,
    }


# =====================================================================================
# 5 — GENERATE 2,000 EPISODES WITH EXISTING FROZEN QC GATES
# =====================================================================================

accepted = []
rejections = []
episode_rows = []

for i, row in enumerate(manifest.itertuples(index=False)):

    selected = None
    selected_metrics = None
    reason = "NO_ATTEMPTS"

    for attempt in range(MAX_ATTEMPTS):

        attempt_seed = (
            int(row.Seed)
            + 100003 * attempt
        )

        candidate = generate_episode_v2(
            split=row.Split,
            family=row.Family,
            seed=attempt_seed,
            hard=bool(row.Hard_Benign),
            source=row.Source
        )

        passed, reason, metrics = gate(
            candidate,
            row.Family
        )

        if passed:
            selected = candidate
            selected_metrics = metrics
            break

        rejections.append({
            "Episode_ID": row.Episode_ID,
            "Attempt": attempt + 1,
            "Failure": reason
        })

    if selected is None:

        pd.DataFrame(rejections).to_csv(
            OUT_CORRECTED / "generation_failures.csv",
            index=False
        )

        raise RuntimeError(
            f"Episode {row.Episode_ID} failed after "
            f"{MAX_ATTEMPTS} attempts. Reason: {reason}"
        )

    values = selected["features"].astype(
        np.float32
    )

    episode_df = pd.DataFrame(
        values,
        columns=FEATURES
    )

    episode_df.insert(
        0,
        "Minute",
        np.arange(EPISODE_MINUTES)
    )

    episode_df.insert(
        0,
        "Episode_ID",
        row.Episode_ID
    )

    episode_rows.append(episode_df)

    accepted.append({
        "Episode_ID": row.Episode_ID,
        "Split": row.Split,
        "Family": row.Family if row.Family is not None else "BENIGN",
        "Hard_Benign": bool(row.Hard_Benign),
        "Source": selected["source"],
        "Calibration_Group_ID": selected["group_id"],
        "Calibration_Block_IDs": selected["block_ids"],
        "Onset_Minute": selected["onset"],
        "Precursor_Length": selected["precursor_len"],
        "Benign_Anomaly_Start": selected["benign_anomaly_start"],
        "Seed": row.Seed,
        "Attempts": attempt + 1,
        **selected_metrics
    })

    if (i + 1) % 200 == 0:
        print(
            f"ACCEPTED {i+1:,} / {len(manifest):,}"
        )

meta = pd.DataFrame(accepted)

minutes = pd.concat(
    episode_rows,
    ignore_index=True
)

print("\nGeneration completed.")
print("Accepted:", len(meta))
print("Rejected proposals:", len(rejections))


# =====================================================================================
# 6 — STRUCTURAL AND PHYSICAL DATA QUALITY CHECKS
# =====================================================================================

assert len(meta) == 2000
assert len(minutes) == 2000 * EPISODE_MINUTES

assert meta.Episode_ID.is_unique
assert not minutes.duplicated(
    ["Episode_ID", "Minute"]
).any()

attack_meta = meta[
    meta.Family != "BENIGN"
]

benign_meta = meta[
    meta.Family == "BENIGN"
]

assert len(attack_meta) == 1500
assert len(benign_meta) == 500

assert benign_meta.Hard_Benign.sum() == 250

assert attack_meta.Onset_Minute.between(
    ONSET_LOW,
    ONSET_HIGH
).all()

assert attack_meta.Onset_Minute.nunique() > 20

assert attack_meta.Precursor_Length.between(
    15, 30
).all()

assert (
    attack_meta.signal_late
    >
    attack_meta.signal_early
).all()

assert (
    attack_meta.signal_attack
    >
    attack_meta.signal_late
).all()

hard_meta = benign_meta[
    benign_meta.Hard_Benign
]

assert hard_meta.Benign_Anomaly_Start.nunique() > 20

for family in FAMILIES:
    assert (meta.Family == family).sum() == 300

# Calibration-group isolation
group_leakage = (
    meta.groupby(
        ["Source", "Calibration_Group_ID"]
    )
    .Split.nunique()
)

assert group_leakage.max() == 1

# Numerical checks
numeric = minutes[FEATURES].to_numpy(float)

assert np.isfinite(numeric).all()
assert (numeric >= 0).all()

assert np.allclose(
    minutes.Total_Packets,
    minutes.Forward_Packets_Sum
    + minutes.Backward_Packets_Sum,
    rtol=1e-5,
    atol=1e-3
)

assert np.allclose(
    minutes.Total_Bytes,
    minutes.Forward_Bytes_Sum
    + minutes.Backward_Bytes_Sum,
    rtol=1e-5,
    atol=1e-2
)

assert (
    minutes.Flow_Duration_Max
    >= minutes.Flow_Duration_Mean
).all()

# Exact temporal continuity
for _, group in minutes.groupby(
    "Episode_ID",
    sort=False
):
    assert np.array_equal(
        group.Minute.to_numpy(),
        np.arange(EPISODE_MINUTES)
    )

print("\nSTRUCTURAL QUALITY CHECKS: PASS")


# =====================================================================================
# 7 — BUILD CAUSAL FORECASTING WINDOWS
# =====================================================================================

window_report = {}

for split in SPLIT_COUNTS:

    X = []
    keys = []
    onset_targets = []
    family_targets = []
    horizon_targets = []

    subset = meta[
        meta.Split == split
    ]

    minute_lookup = {
        episode_id: group[FEATURES].to_numpy(np.float32)
        for episode_id, group in minutes[
            minutes.Episode_ID.isin(subset.Episode_ID)
        ].groupby("Episode_ID", sort=False)
    }

    for entry in subset.itertuples(index=False):

        z = minute_lookup[entry.Episode_ID]

        onset = int(entry.Onset_Minute)

        for t in range(W - 1, EPISODE_MINUTES):

            # No forecasting from attack-state observations.
            if onset >= 0 and t >= onset:
                break

            # Require full observation horizon.
            if t + MAX_H >= EPISODE_MINUTES:
                continue

            future = (
                onset - t
                if onset >= 0
                else -1
            )

            X.append(
                z[t-W+1:t+1]
            )

            keys.append(
                (entry.Episode_ID, t)
            )

            positive = (
                1 <= future <= MAX_H
            )

            onset_targets.append(
                future if positive else 0
            )

            family_targets.append(
                FAMILIES.index(entry.Family) + 1
                if positive else 0
            )

            horizon_targets.append(
                [
                    int(1 <= future <= h)
                    for h in HORIZONS
                ]
            )

    X = np.stack(X).astype(np.float32)

    onset_targets = np.asarray(
        onset_targets,
        dtype=np.int16
    )

    family_targets = np.asarray(
        family_targets,
        dtype=np.int8
    )

    horizon_targets = np.asarray(
        horizon_targets,
        dtype=np.int8
    )

    assert X.shape[1:] == (
        W,
        len(FEATURES)
    )

    assert np.isfinite(X).all()

    assert onset_targets.min() >= 0
    assert onset_targets.max() <= MAX_H

    # Competing-risk target consistency
    assert np.array_equal(
        onset_targets > 0,
        family_targets > 0
    )

    for h_index, h in enumerate(HORIZONS):

        expected = (
            (onset_targets >= 1)
            &
            (onset_targets <= h)
        ).astype(np.int8)

        assert np.array_equal(
            horizon_targets[:, h_index],
            expected
        )

    np.savez_compressed(
        OUT_CORRECTED / f"windows_W10_{split}.npz",
        X=X,
        onset_class=onset_targets,
        family_class=family_targets,
        y_horizons=horizon_targets,
        episode_id=np.asarray(
            [v[0] for v in keys]
        ),
        minute=np.asarray(
            [v[1] for v in keys],
            dtype=np.int16
        )
    )

    window_report[split] = {
        "shape": list(X.shape),
        "horizon_prevalence": (
            horizon_targets.mean(axis=0).tolist()
        ),
        "episodes": len(subset)
    }

    print(
        split,
        X.shape,
        "H5/H10/H15 prevalence:",
        horizon_targets.mean(axis=0).round(4)
    )


# =====================================================================================
# 8 — TRAIN-ONLY TEMPORAL SHORTCUT DIAGNOSTIC
# =====================================================================================

train_pack = np.load(
    OUT_CORRECTED / "windows_W10_TRAIN.npz",
    allow_pickle=False
)

y = train_pack["y_horizons"][:, 1].astype(int)

position = train_pack["minute"].astype(int)

episode_ids = train_pack["episode_id"].astype(str)

source_by_episode = (
    meta.set_index("Episode_ID")["Source"]
)

sources = source_by_episode.loc[
    episode_ids
].to_numpy()

source_codes = (
    pd.Series(sources)
    .astype("category")
    .cat.codes
    .to_numpy()
)

shortcut_features = np.column_stack(
    [position, source_codes]
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED
)

train_idx, check_idx = next(
    splitter.split(
        shortcut_features,
        y,
        groups=episode_ids
    )
)

shortcut_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    )
)

shortcut_model.fit(
    shortcut_features[train_idx],
    y[train_idx]
)

probabilities = shortcut_model.predict_proba(
    shortcut_features[check_idx]
)[:, 1]

prevalence = float(
    y[check_idx].mean()
)

shortcut_ap = float(
    average_precision_score(
        y[check_idx],
        probabilities
    )
)

shortcut_auc = float(
    roc_auc_score(
        y[check_idx],
        probabilities
    )
)

shortcut_report = {
    "scope": "TRAIN_ONLY_EPISODE_DISJOINT",
    "task": "H10",
    "positive_prevalence": prevalence,
    "minute_source_only_AP": shortcut_ap,
    "minute_source_only_ROC_AUC": shortcut_auc,
    "AP_over_prevalence": (
        shortcut_ap / max(prevalence, 1e-12)
    ),
    "unique_attack_onsets": int(
        attack_meta.Onset_Minute.nunique()
    ),
    "unique_benign_event_starts": int(
        hard_meta.Benign_Anomaly_Start.nunique()
    )
}

print("\nTRAIN-ONLY SHORTCUT DIAGNOSTIC")
print(json.dumps(shortcut_report, indent=2))

# This is a diagnostic, not a final proof against shortcut learning.
# Do not inspect TEST predictions here.


# =====================================================================================
# 9 — SAVE ARTIFACTS AND INTEGRITY HASHES
# =====================================================================================

minutes.to_parquet(
    OUT_CORRECTED / "minute_predictors.parquet",
    index=False
)

meta.to_csv(
    OUT_CORRECTED / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv",
    index=False
)

pd.DataFrame(rejections).to_csv(
    OUT_CORRECTED / "qc_rejections.csv",
    index=False
)

contract = {
    "dataset": "Synthetic_NAF_v2",
    "status": "CORRECTED_RELEASE_CANDIDATE",
    "generator_seed": SEED,
    "episode_count": 2000,
    "attack_episodes": 1500,
    "benign_episodes": 500,
    "features": FEATURES,
    "attack_families": FAMILIES,
    "history_minutes": W,
    "forecast_horizons": HORIZONS,
    "onset_range": [ONSET_LOW, ONSET_HIGH],
    "precursor_range": [15, 30],
    "sampling_cadence_minutes": 1,
    "baseline_method": (
        "Group-isolated log-space interpolation "
        "with temporally correlated noise"
    ),
    "source_conditioning": True,
    "empirically_observed_precursors": False,
    "limitations": [
        "Attack precursor dynamics are simulated",
        "Some families exist in only one source",
        "Calibration groups are reused within splits",
        "Real-world predictive validity is not established"
    ],
    "quality_thresholds": {
        "late_minus_early": QC_MIN_LATE_VS_EARLY,
        "attack_minus_late": QC_MIN_ATTACK_VS_PRE,
        "max_precursor_to_attack": QC_MAX_PRE_TO_ATTACK
    }
}

(OUT_CORRECTED / "FEATURE_AND_DATASET_CONTRACT.json").write_text(
    json.dumps(contract, indent=2)
)

qc_summary = {
    "status": "DATA_ONLY_CHECKS_PASSED",
    "accepted_episodes": len(meta),
    "rejected_proposals": len(rejections),
    "splits": (
        meta.groupby("Split").size().to_dict()
    ),
    "windows": window_report
}

(OUT_CORRECTED / "QC_SUMMARY.json").write_text(
    json.dumps(qc_summary, indent=2)
)

(OUT_CORRECTED / "TRAIN_ONLY_SHORTCUT_DIAGNOSTICS.json").write_text(
    json.dumps(shortcut_report, indent=2)
)

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

hashes = {
    path.name: sha256_file(path)
    for path in sorted(OUT_CORRECTED.iterdir())
    if path.is_file()
    and path.name != "SHA256SUMS.json"
}

(OUT_CORRECTED / "SHA256SUMS.json").write_text(
    json.dumps(hashes, indent=2)
)

print("\n" + "=" * 90)
print("CORRECTED SYNTHETIC_NAF_V2 — GENERATION COMPLETE")
print("=" * 90)

print("Episodes:", len(meta))
print("Attack episodes:", len(attack_meta))
print("Benign episodes:", len(benign_meta))
print("Rejected proposals:", len(rejections))

print("\nAttack onset range:")
print(
    attack_meta.Onset_Minute.describe()
)

print("\nSource/family distribution:")
print(
    pd.crosstab(
        meta.Family,
        meta.Source
    )
)

print("\nForecasting window summary:")
print(
    json.dumps(window_report, indent=2)
)

print("\nShortcut diagnostic:")
print(
    json.dumps(shortcut_report, indent=2)
)

print("\nOutput folder:")
print(OUT_CORRECTED)

print("\nIMPORTANT:")
print("This is a corrected synthetic RELEASE CANDIDATE.")
print("Data-only QC passed; scientific suitability remains to be assessed.")
print("TEST model performance has not been evaluated.")

In [ ]:

# =============================================================================
# SYNTHETIC_NAF_V2 — DECISION-TIME MATCHED FORECASTING DIAGNOSTIC
#
# PURPOSE
# 1. Use only the corrected V2 TRAIN partition.
# 2. Create an internal split by calibration GROUP, not random windows.
# 3. Match H10 positives and negatives on exact minute + source.
# 4. Compare shortcut-only vs network-only forecasting.
# 5. Save matched diagnostic samples separately.
# 6. Never modify official TRAIN / VALIDATION / TEST data.
#
# This is NOT the final forecasting model benchmark.
# =============================================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    balanced_accuracy_score,
    brier_score_loss
)

# =============================================================================
# A — PATHS
# =============================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATA_DIR = (
    ROOT
    / "Synthetic_NAF_v2"
    / "release_candidate_corrected"
)

OUT = (
    ROOT
    / "Synthetic_NAF_v2"
    / "decision_time_matched_diagnostic"
)

OUT.mkdir(parents=True, exist_ok=True)

REQUIRED = [
    DATA_DIR / "windows_W10_TRAIN.npz",
    DATA_DIR / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv",
    DATA_DIR / "FEATURE_AND_DATASET_CONTRACT.json"
]

for path in REQUIRED:
    if not path.exists():
        raise FileNotFoundError(path)

print("=" * 95)
print("V2 — DECISION-TIME MATCHED FORECASTING DIAGNOSTIC")
print("=" * 95)

# =============================================================================
# B — LOAD TRAIN ONLY
# =============================================================================

SEED = 20261022
rng = np.random.default_rng(SEED)

pack = np.load(
    DATA_DIR / "windows_W10_TRAIN.npz",
    allow_pickle=False
)

X = pack["X"]
Y = pack["y_horizons"]

episode_ids = (
    pd.Series(pack["episode_id"])
    .astype(str)
    .to_numpy()
)

minutes = pack["minute"].astype(int)

assert X.ndim == 3
assert X.shape[1:] == (10, 14)
assert Y.shape == (len(X), 3)

# H10 is the middle horizon: H5, H10, H15
y = Y[:, 1].astype(np.int8)

meta = pd.read_csv(
    DATA_DIR / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
)

meta["Episode_ID"] = meta["Episode_ID"].astype(str)

assert meta["Episode_ID"].is_unique
assert set(meta["Split"].unique()) == {
    "TRAIN", "VALIDATION", "TEST"
}

train_meta = meta[
    meta["Split"] == "TRAIN"
].copy()

lookup = train_meta.set_index("Episode_ID")

unknown = set(episode_ids) - set(lookup.index)

if unknown:
    raise RuntimeError(
        f"Unknown TRAIN episode IDs: {list(unknown)[:5]}"
    )

sources = lookup.loc[
    episode_ids, "Source"
].astype(str).to_numpy()

calibration_groups = lookup.loc[
    episode_ids, "Calibration_Group_ID"
].astype(str).to_numpy()

# These group IDs must be disjoint between
# internal fitting and diagnostic holdout.
group_keys = np.array([
    f"{s}::{g}"
    for s, g in zip(sources, calibration_groups)
])

frame = pd.DataFrame({
    "row_index": np.arange(len(y)),
    "Episode_ID": episode_ids,
    "Minute": minutes,
    "Source": sources,
    "Calibration_Group": group_keys,
    "H10": y
})

assert np.isfinite(X).all()
assert not frame.duplicated(
    ["Episode_ID", "Minute"]
).any()

print("\nOriginal TRAIN windows:", X.shape)
print("Original H10 prevalence:", round(float(y.mean()), 5))
print("TRAIN calibration groups:", len(np.unique(group_keys)))

# =============================================================================
# C — INTERNAL GROUP-DISJOINT SPLIT
# =============================================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED
)

fit_idx, holdout_idx = next(
    splitter.split(
        frame,
        frame["H10"],
        groups=frame["Calibration_Group"]
    )
)

fit_frame = frame.iloc[fit_idx].copy()
holdout_frame = frame.iloc[holdout_idx].copy()

fit_groups = set(
    fit_frame["Calibration_Group"]
)

holdout_groups = set(
    holdout_frame["Calibration_Group"]
)

assert fit_groups.isdisjoint(holdout_groups)

assert set(
    fit_frame["Episode_ID"]
).isdisjoint(
    set(holdout_frame["Episode_ID"])
)

print("\nINTERNAL PARTITION")
print("Fitting rows:", len(fit_frame))
print("Holdout rows:", len(holdout_frame))
print("Fitting groups:", len(fit_groups))
print("Holdout groups:", len(holdout_groups))

# =============================================================================
# D — MATCH POSITIVES AND NEGATIVES
#
# For each (Source, Minute) stratum:
#   N positive H10 samples
#   N negative H10 samples
#
# Selection is without replacement.
# Strata that cannot provide both classes are excluded.
#
# No model predictions are used for matching.
# =============================================================================

def make_matched_table(input_frame, seed):

    local_rng = np.random.default_rng(seed)

    selected = []
    strata_report = []

    for (source, minute), group in input_frame.groupby(
        ["Source", "Minute"],
        sort=True
    ):

        positive = group[
            group["H10"] == 1
        ]

        negative = group[
            group["H10"] == 0
        ]

        n = min(
            len(positive),
            len(negative)
        )

        if n == 0:
            continue

        pos_sample = local_rng.choice(
            positive.index.to_numpy(),
            size=n,
            replace=False
        )

        neg_sample = local_rng.choice(
            negative.index.to_numpy(),
            size=n,
            replace=False
        )

        selected.extend(pos_sample.tolist())
        selected.extend(neg_sample.tolist())

        strata_report.append({
            "Source": source,
            "Minute": int(minute),
            "Positive_Count": int(n),
            "Negative_Count": int(n)
        })

    if not selected:
        raise RuntimeError(
            "No source/minute strata contain both classes."
        )

    matched = input_frame.loc[selected].copy()

    matched = matched.sample(
        frac=1.0,
        random_state=seed
    ).reset_index(drop=True)

    assert matched["H10"].nunique() == 2

    counts = matched.groupby(
        ["Source", "Minute", "H10"]
    ).size().unstack(fill_value=0)

    assert (counts[0] == counts[1]).all()

    assert matched["row_index"].is_unique

    return matched, pd.DataFrame(strata_report)


matched_fit, strata_fit = make_matched_table(
    fit_frame,
    SEED + 1
)

matched_holdout, strata_holdout = make_matched_table(
    holdout_frame,
    SEED + 2
)

assert set(
    matched_fit["Calibration_Group"]
).isdisjoint(
    set(matched_holdout["Calibration_Group"])
)

print("\nMATCHED SAMPLE COUNTS")

for name, table in [
    ("FIT", matched_fit),
    ("HOLDOUT", matched_holdout)
]:
    print(
        name,
        "rows =", len(table),
        "positives =", int(table["H10"].sum()),
        "prevalence =", round(table["H10"].mean(), 4)
    )

# =============================================================================
# E — CONSTRUCT TWO INDEPENDENT FEATURE VIEWS
#
# Shortcut model:
#   Minute + Source
#
# Network model:
#   Only flattened W10 x 14 predictors
#
# No episode IDs, metadata, source, minute or
# future labels enter the network model.
# =============================================================================

fit_rows = matched_fit["row_index"].to_numpy(int)
check_rows = matched_holdout["row_index"].to_numpy(int)

y_fit = y[fit_rows]
y_check = y[check_rows]

# Shortcut-only feature matrix
source_categories = sorted(frame["Source"].unique())

def make_shortcut_features(table):

    minute_column = table["Minute"].to_numpy(
        dtype=float
    ).reshape(-1, 1)

    source_dummies = np.column_stack([
        (table["Source"].to_numpy() == src).astype(float)
        for src in source_categories
    ])

    return np.column_stack([
        minute_column,
        source_dummies
    ])

shortcut_fit = make_shortcut_features(matched_fit)
shortcut_check = make_shortcut_features(matched_holdout)

# Network-only temporal data
network_fit = X[fit_rows].reshape(
    len(fit_rows), -1
)

network_check = X[check_rows].reshape(
    len(check_rows), -1
)

assert network_fit.shape[1] == 140
assert network_check.shape[1] == 140

# =============================================================================
# F — TRAIN DIAGNOSTIC BASELINES
#
# Fit once on internal TRAIN fitting groups.
# Evaluate on internal TRAIN held-out groups.
#
# These are diagnostic baselines, not official models.
# =============================================================================

def evaluate(name, x_fit, x_check):

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=0.1,
            max_iter=1500,
            solver="lbfgs",
            random_state=SEED
        )
    )

    model.fit(x_fit, y_fit)

    prob = model.predict_proba(x_check)[:, 1]

    result = {
        "Model": name,
        "AP": float(average_precision_score(y_check, prob)),
        "ROC_AUC": float(roc_auc_score(y_check, prob)),
        "Balanced_Accuracy": float(
            balanced_accuracy_score(
                y_check,
                (prob >= 0.5).astype(int)
            )
        ),
        "Brier": float(brier_score_loss(y_check, prob))
    }

    return result


results = []

results.append(
    evaluate(
        "Minute_And_Source_Only",
        shortcut_fit,
        shortcut_check
    )
)

results.append(
    evaluate(
        "Network_W10_Only",
        network_fit,
        network_check
    )
)

results_df = pd.DataFrame(results)

baseline_ap = float(y_check.mean())

print("\n" + "=" * 95)
print("TIME-MATCHED INTERNAL HOLDOUT RESULTS")
print("=" * 95)

print("\nMatched positive prevalence:", baseline_ap)
print(results_df.to_string(index=False))

# =============================================================================
# G — INTERPRETATION
# =============================================================================

shortcut_ap = results_df.loc[
    results_df["Model"] == "Minute_And_Source_Only",
    "AP"
].iloc[0]

shortcut_auc = results_df.loc[
    results_df["Model"] == "Minute_And_Source_Only",
    "ROC_AUC"
].iloc[0]

network_ap = results_df.loc[
    results_df["Model"] == "Network_W10_Only",
    "AP"
].iloc[0]

network_auc = results_df.loc[
    results_df["Model"] == "Network_W10_Only",
    "ROC_AUC"
].iloc[0]

print("\nINTERPRETATION")

print(
    f"Shortcut AP: {shortcut_ap:.4f} "
    f"(baseline {baseline_ap:.4f})"
)

print(
    f"Shortcut ROC-AUC: {shortcut_auc:.4f}"
)

print(
    f"Network AP: {network_ap:.4f}"
)

print(
    f"Network ROC-AUC: {network_auc:.4f}"
)

if shortcut_auc > 0.55:
    print(
        "\nWARNING: Residual decision-time/source "
        "shortcut signal remains."
    )
else:
    print(
        "\nCHECK: The shortcut-only classifier "
        "is near chance discrimination."
    )

if network_auc > shortcut_auc + 0.05:
    print(
        "CHECK: Network features add predictive "
        "discrimination beyond the shortcut baseline."
    )
else:
    print(
        "WARNING: Network feature advantage "
        "is not clearly established."
    )

print(
    "\nThese are descriptive checks, not "
    "statistical significance tests or "
    "proof of real-world validity."
)

# =============================================================================
# H — SAVE MATCHED DIAGNOSTIC DATA
#
# We store row indices, labels and grouping keys.
# The source TRAIN arrays remain the single source
# of truth, preventing duplicate dataset versions.
# =============================================================================

matched_fit.to_csv(
    OUT / "MATCHED_INTERNAL_FIT.csv",
    index=False
)

matched_holdout.to_csv(
    OUT / "MATCHED_INTERNAL_HOLDOUT.csv",
    index=False
)

strata_fit.to_csv(
    OUT / "MATCHED_FIT_STRATA.csv",
    index=False
)

strata_holdout.to_csv(
    OUT / "MATCHED_HOLDOUT_STRATA.csv",
    index=False
)

results_df.to_csv(
    OUT / "SHORTCUT_VS_NETWORK_RESULTS.csv",
    index=False
)

report = {
    "dataset": "Synthetic_NAF_v2",
    "candidate": "release_candidate_corrected",
    "scope": "TRAIN_ONLY",
    "task": "H10",
    "history": "W10",
    "matching": "exact_source_and_minute",
    "internal_split": "calibration_group_disjoint",
    "fit_rows": int(len(matched_fit)),
    "holdout_rows": int(len(matched_holdout)),
    "matched_prevalence": baseline_ap,
    "original_train_prevalence": float(y.mean()),
    "models": results_df.to_dict(orient="records"),
    "limitations": [
        "Matched prevalence is artificial and must not be used for operational precision estimates.",
        "Matching does not eliminate feature-based generator shortcuts.",
        "Only H10 is tested in this diagnostic.",
        "Diagnostic models are not the final forecasting models.",
        "No official VALIDATION or TEST model evaluation was performed."
    ]
}

(OUT / "DIAGNOSTIC_REPORT.json").write_text(
    json.dumps(report, indent=2)
)

def file_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


hashes = {
    path.name: file_sha256(path)
    for path in sorted(OUT.iterdir())
    if path.is_file()
    and path.name != "SHA256SUMS.json"
}

(OUT / "SHA256SUMS.json").write_text(
    json.dumps(hashes, indent=2)
)

print("\n" + "=" * 95)
print("DIAGNOSTIC COMPLETE")
print("=" * 95)
print("Saved to:", OUT)
print("Original corrected dataset: UNCHANGED")
print("Official VALIDATION and TEST: NOT USED")


In [ ]:

# ==========================================================================================
# SYNTHETIC_NAF_V2 — FINAL READ-ONLY RELEASE VERIFICATION
#
# Reads:
#   Synthetic_NAF_v2/release_candidate_corrected/
#
# Writes ONLY to:
#   Synthetic_NAF_v2/final_release_verification/
#
# DOES NOT:
#   - Regenerate or modify dataset episodes
#   - Change forecasting labels
#   - Train or evaluate ML models
#   - Touch the original dataset audit
#   - Unlock or score the TEST partition
#   - Automatically declare the dataset frozen
#
# ==========================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd

# ==========================================================================================
# 1 — PATHS
# ==========================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATA = (
    ROOT
    / "Synthetic_NAF_v2"
    / "release_candidate_corrected"
)

REPORT_DIR = (
    ROOT
    / "Synthetic_NAF_v2"
    / "final_release_verification"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 105)
print("SYNTHETIC_NAF_V2 — FINAL RELEASE VERIFICATION")
print("=" * 105)

print("Dataset:", DATA)
print("Report directory:", REPORT_DIR)

if not DATA.exists():
    raise FileNotFoundError(
        f"Corrected release candidate not found: {DATA}"
    )

# ==========================================================================================
# 2 — LOAD FROZEN CONTRACT
# ==========================================================================================

CONTRACT_PATH = DATA / "FEATURE_AND_DATASET_CONTRACT.json"

with open(CONTRACT_PATH, "r") as f:
    contract = json.load(f)

FEATURES = contract["features"]
FAMILIES = contract["attack_families"]

W = int(contract["history_minutes"])
HORIZONS = list(map(int, contract["forecast_horizons"]))

MAX_H = max(HORIZONS)
EPISODE_MINUTES = 110

SPLITS = ["TRAIN", "VALIDATION", "TEST"]

EXPECTED_COUNTS = {
    "TRAIN": 1400,
    "VALIDATION": 300,
    "TEST": 300
}

EXPECTED_FAMILY_COUNTS = {
    "TRAIN": 210,
    "VALIDATION": 45,
    "TEST": 45
}

EXPECTED_BENIGN_COUNTS = {
    "TRAIN": 350,
    "VALIDATION": 75,
    "TEST": 75
}

# Independent contract expectations.
EXPECTED_FEATURES = [
    "Flow_Count",
    "Flow_Duration_Mean",
    "Flow_Duration_Std",
    "Flow_Duration_Max",
    "Forward_Packets_Sum",
    "Backward_Packets_Sum",
    "Total_Packets",
    "Forward_Bytes_Sum",
    "Backward_Bytes_Sum",
    "Total_Bytes",
    "Packets_Per_Second",
    "Bytes_Per_Second",
    "Forward_Backward_Packet_Ratio",
    "Forward_Backward_Byte_Ratio"
]

EXPECTED_FAMILIES = [
    "BRUTE_AUTH",
    "DOS_DDOS",
    "EXPLOIT_APP",
    "FUZZING",
    "RECON_SCAN"
]

# ==========================================================================================
# 3 — CHECK TRACKER
# ==========================================================================================

checks = []

def check(name, condition, details="", severity="BLOCKER"):

    passed = bool(condition)

    checks.append({
        "Check": name,
        "Status": "PASS" if passed else "FAIL",
        "Severity": severity,
        "Details": str(details)
    })

    symbol = "PASS" if passed else "FAIL"

    print(f"[{symbol}] {name}")

    if not passed:
        print("       ", details)

    return passed


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def save_json(path, obj):

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            default=str
        )
    )


check(
    "Feature schema and order",
    FEATURES == EXPECTED_FEATURES,
    f"Actual features: {FEATURES}"
)

check(
    "Attack-family contract",
    FAMILIES == EXPECTED_FAMILIES,
    f"Actual families: {FAMILIES}"
)

check(
    "History and horizon contract",
    W == 10 and HORIZONS == [5, 10, 15],
    f"W={W}, horizons={HORIZONS}"
)

# ==========================================================================================
# 4 — REQUIRED ARTIFACTS AND INTEGRITY HASHES
# ==========================================================================================

required_files = [
    "minute_predictors.parquet",
    "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv",
    "FEATURE_AND_DATASET_CONTRACT.json",
    "QC_SUMMARY.json",
    "qc_rejections.csv",
    "SHA256SUMS.json",
    "windows_W10_TRAIN.npz",
    "windows_W10_VALIDATION.npz",
    "windows_W10_TEST.npz"
]

missing = [
    name
    for name in required_files
    if not (DATA / name).exists()
]

if missing:
    raise FileNotFoundError(
        f"Required dataset artifacts missing: {missing}"
    )

with open(DATA / "SHA256SUMS.json", "r") as f:
    recorded_hashes = json.load(f)

actual_hashes = {}

hash_failures = []

for filename in required_files:

    if filename == "SHA256SUMS.json":
        continue

    actual = sha256_file(DATA / filename)
    actual_hashes[filename] = actual

    expected = recorded_hashes.get(filename)

    if actual != expected:
        hash_failures.append(filename)

check(
    "Required artifact SHA256 integrity",
    len(hash_failures) == 0,
    f"Hash failures: {hash_failures}"
)

extra_recorded = sorted(
    set(recorded_hashes) -
    {
        p.name for p in DATA.iterdir()
        if p.is_file() and p.name != "SHA256SUMS.json"
    }
)

check(
    "Recorded hash file coverage",
    len(extra_recorded) == 0,
    f"Missing recorded files: {extra_recorded}"
)

# ==========================================================================================
# 5 — READ METADATA AND MINUTE PREDICTORS
# ==========================================================================================

meta = pd.read_csv(
    DATA / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
)

minutes = pd.read_parquet(
    DATA / "minute_predictors.parquet"
)

meta["Episode_ID"] = meta["Episode_ID"].astype(str)
minutes["Episode_ID"] = minutes["Episode_ID"].astype(str)

check(
    "Exactly 2,000 episodes",
    len(meta) == 2000,
    f"Actual: {len(meta)}"
)

check(
    "Unique episode identifiers",
    meta.Episode_ID.is_unique
)

check(
    "Exactly 1,500 attack episodes",
    int((meta.Family != "BENIGN").sum()) == 1500
)

check(
    "Exactly 500 benign episodes",
    int((meta.Family == "BENIGN").sum()) == 500
)

check(
    "Exactly 250 hard-benign episodes",
    int(
        meta.loc[
            meta.Family == "BENIGN",
            "Hard_Benign"
        ].astype(bool).sum()
    ) == 250
)

check(
    "All five attack families present",
    set(meta.Family.unique()) ==
    set(FAMILIES + ["BENIGN"])
)

# ==========================================================================================
# 6 — EPISODE / FAMILY / SPLIT BALANCE
# ==========================================================================================

for split in SPLITS:

    subset = meta[meta.Split == split]

    check(
        f"{split} episode count",
        len(subset) == EXPECTED_COUNTS[split],
        f"Actual: {len(subset)}"
    )

    for family in FAMILIES:

        count = int(
            (subset.Family == family).sum()
        )

        check(
            f"{split} / {family}",
            count == EXPECTED_FAMILY_COUNTS[split],
            f"Actual: {count}"
        )

    benign_count = int(
        (subset.Family == "BENIGN").sum()
    )

    check(
        f"{split} benign count",
        benign_count == EXPECTED_BENIGN_COUNTS[split],
        f"Actual: {benign_count}"
    )

# ==========================================================================================
# 7 — CALIBRATION GROUP ISOLATION
# ==========================================================================================

group_ownership = (
    meta.groupby(
        ["Source", "Calibration_Group_ID"]
    )
    .Split.nunique()
)

check(
    "Calibration groups isolated across splits",
    bool((group_ownership == 1).all()),
    f"Maximum split ownership: {group_ownership.max()}"
)

check(
    "No unknown split names",
    set(meta.Split.unique()) == set(SPLITS)
)

# ==========================================================================================
# 8 — MINUTE DATA INTEGRITY
# ==========================================================================================

missing_features = [
    f for f in FEATURES
    if f not in minutes.columns
]

check(
    "All predictor columns available",
    len(missing_features) == 0,
    f"Missing: {missing_features}"
)

if missing_features:
    raise RuntimeError(
        "Cannot verify numerical data without all predictors."
    )

check(
    "Minute-row count",
    len(minutes) == 2000 * EPISODE_MINUTES,
    f"Actual: {len(minutes)}"
)

check(
    "Unique episode/minute keys",
    not minutes.duplicated(
        ["Episode_ID", "Minute"]
    ).any()
)

minute_groups = minutes.groupby(
    "Episode_ID",
    sort=False
)

lengths = minute_groups.size()

check(
    "Exactly 110 minutes per episode",
    len(lengths) == 2000
    and bool((lengths == 110).all())
)

minute_key_valid = (
    minutes["Minute"].between(0, 109).all()
    and
    minute_groups["Minute"].nunique().eq(110).all()
)

check(
    "Complete minute index 0 to 109",
    bool(minute_key_valid)
)

numeric = minutes[FEATURES].to_numpy(
    dtype=np.float64
)

check(
    "All predictor values finite",
    bool(np.isfinite(numeric).all())
)

check(
    "No negative predictor values",
    bool((numeric >= 0).all())
)

def close(a, b, atol=1e-3):

    return bool(
        np.allclose(
            a,
            b,
            rtol=1e-5,
            atol=atol
        )
    )


check(
    "Packet-total identity",
    close(
        minutes.Total_Packets,
        minutes.Forward_Packets_Sum
        + minutes.Backward_Packets_Sum
    )
)

check(
    "Byte-total identity",
    close(
        minutes.Total_Bytes,
        minutes.Forward_Bytes_Sum
        + minutes.Backward_Bytes_Sum,
        atol=1e-2
    )
)

check(
    "Packets-per-second consistency",
    close(
        minutes.Packets_Per_Second,
        minutes.Total_Packets / 60.0
    )
)

check(
    "Bytes-per-second consistency",
    close(
        minutes.Bytes_Per_Second,
        minutes.Total_Bytes / 60.0
    )
)

check(
    "Packet direction-ratio consistency",
    close(
        minutes.Forward_Backward_Packet_Ratio,
        minutes.Forward_Packets_Sum
        /
        np.maximum(minutes.Backward_Packets_Sum, 1.0)
    )
)

check(
    "Byte direction-ratio consistency",
    close(
        minutes.Forward_Backward_Byte_Ratio,
        minutes.Forward_Bytes_Sum
        /
        np.maximum(minutes.Backward_Bytes_Sum, 1.0)
    )
)

check(
    "Flow-duration ordering",
    bool(
        (
            minutes.Flow_Duration_Max
            >= minutes.Flow_Duration_Mean
        ).all()
    )
)

# ==========================================================================================
# 9 — ATTACK TIMING AND PRECURSOR DESIGN
# ==========================================================================================

attacks = meta[
    meta.Family != "BENIGN"
].copy()

benign = meta[
    meta.Family == "BENIGN"
].copy()

check(
    "Attack onset range 46-82",
    bool(
        attacks.Onset_Minute.between(46, 82).all()
    )
)

check(
    "At least 20 unique attack onset minutes",
    attacks.Onset_Minute.nunique() >= 20,
    f"Unique: {attacks.Onset_Minute.nunique()}"
)

check(
    "Precursor durations 15-30 minutes",
    bool(
        attacks.Precursor_Length.between(15, 30).all()
    )
)

check(
    "Benign episodes have no attack onset",
    bool((benign.Onset_Minute == -1).all())
)

required_scores = [
    "signal_early",
    "signal_late",
    "signal_attack"
]

missing_scores = [
    c for c in required_scores
    if c not in attacks.columns
]

check(
    "Recorded precursor-score metadata",
    len(missing_scores) == 0,
    f"Missing: {missing_scores}"
)

if not missing_scores:

    check(
        "Recorded late precursor stronger than early",
        bool(
            (
                attacks.signal_late
                >
                attacks.signal_early
            ).all()
        )
    )

    check(
        "Recorded attack signal stronger than late precursor",
        bool(
            (
                attacks.signal_attack
                >
                attacks.signal_late
            ).all()
        )
    )

# ==========================================================================================
# 10 — PRECURSOR EVIDENCE / BENIGN-OVERLAP DESCRIPTIVES
#
# This section describes observable variation.
#
# IMPORTANT:
# Without the unmodified counterfactual baseline used
# during generation, we cannot independently reproduce
# the original robust_signal() scores exactly.
#
# No model training occurs.
# ==========================================================================================

primitive_features = [
    "Flow_Count",
    "Flow_Duration_Mean",
    "Flow_Duration_Std",
    "Forward_Packets_Sum",
    "Backward_Packets_Sum",
    "Forward_Bytes_Sum",
    "Backward_Bytes_Sum"
]

primitive_indices = [
    FEATURES.index(f)
    for f in primitive_features
]

all_trajectory_rows = []

# Train-only descriptive reference, avoiding tuning on TEST.
train_minute_ids = set(
    meta.loc[meta.Split == "TRAIN", "Episode_ID"]
)

train_minutes = minutes[
    minutes.Episode_ID.isin(train_minute_ids)
]

source_scales = {}

for source in meta.Source.unique():

    source_episode_ids = set(
        meta.loc[
            (meta.Split == "TRAIN")
            &
            (meta.Source == source)
            &
            (meta.Family == "BENIGN"),
            "Episode_ID"
        ]
    )

    reference = train_minutes[
        train_minutes.Episode_ID.isin(source_episode_ids)
    ][primitive_features].to_numpy(float)

    if len(reference) == 0:
        raise RuntimeError(
            f"No TRAIN benign reference for source {source}"
        )

    transformed = np.log1p(np.maximum(reference, 0))

    med = np.median(transformed, axis=0)

    mad = 1.4826 * np.median(
        np.abs(transformed - med),
        axis=0
    )

    scale = np.maximum(mad, 0.10)

    source_scales[source] = (med, scale)


meta_lookup = meta.set_index("Episode_ID")

for episode_id, group in minute_groups:

    group = group.sort_values("Minute")

    entry = meta_lookup.loc[episode_id]

    arr = group[primitive_features].to_numpy(float)

    transformed = np.log1p(np.maximum(arr, 0))

    # Local reference uses only the episode's first 10 minutes.
    # This is a descriptive reference, not its generator baseline.
    local_baseline = np.median(
        transformed[:10],
        axis=0
    )

    _, scale = source_scales[entry.Source]

    deviation = np.median(
        np.abs(transformed - local_baseline)
        /
        scale,
        axis=1
    )

    onset = int(entry.Onset_Minute)

    if onset >= 0:

        length = int(entry.Precursor_Length)
        start = onset - length

        early = deviation[
            start : min(start + 5, onset)
        ]

        late = deviation[
            max(start, onset - 5) : onset
        ]

        attack = deviation[
            onset : min(onset + 5, len(deviation))
        ]

        # Earlier time before onset and before the precursor.
        pre_start = max(10, start - 15)

        pre_precursor = deviation[
            pre_start:start
        ]

        all_trajectory_rows.append({
            "Episode_ID": episode_id,
            "Split": entry.Split,
            "Source": entry.Source,
            "Family": entry.Family,
            "Early": float(np.median(early)),
            "Late": float(np.median(late)),
            "Attack": float(np.median(attack)),
            "Pre_Precursor": (
                float(np.median(pre_precursor))
                if len(pre_precursor) > 0
                else np.nan
            )
        })

    else:

        # Same clock range as attack development.
        # This is NOT a claim of one-to-one precursor matching.
        early = deviation[35:45]
        late = deviation[55:65]

        all_trajectory_rows.append({
            "Episode_ID": episode_id,
            "Split": entry.Split,
            "Source": entry.Source,
            "Family": "BENIGN_HARD"
            if bool(entry.Hard_Benign)
            else "BENIGN_ORDINARY",
            "Early": float(np.median(early)),
            "Late": float(np.median(late)),
            "Attack": np.nan,
            "Pre_Precursor": float(
                np.median(deviation[15:30])
            )
        })

trajectory = pd.DataFrame(all_trajectory_rows)

train_trajectory = trajectory[
    trajectory.Split == "TRAIN"
]

train_attack = train_trajectory[
    train_trajectory.Family.isin(FAMILIES)
]

train_hard = train_trajectory[
    train_trajectory.Family == "BENIGN_HARD"
]

if len(train_hard) > 0:

    attack_late = train_attack.Late.dropna().to_numpy()
    hard_late = train_hard.Late.dropna().to_numpy()

    hard_low = np.quantile(hard_late, 0.10)
    hard_high = np.quantile(hard_late, 0.90)

    overlap_fraction = float(
        np.mean(
            (attack_late >= hard_low)
            &
            (attack_late <= hard_high)
        )
    )

else:
    overlap_fraction = np.nan

trajectory_summary = {
    "train_attack_early_median": float(
        train_attack.Early.median()
    ),
    "train_attack_late_median": float(
        train_attack.Late.median()
    ),
    "train_attack_post_onset_median": float(
        train_attack.Attack.median()
    ),
    "train_hard_benign_late_median": float(
        train_hard.Late.median()
    ),
    "attack_late_within_hard_benign_p10_p90_fraction": overlap_fraction,
    "method": (
        "Descriptive log-primitive deviation from each "
        "episode's first-ten-minute reference, standardized "
        "with TRAIN benign source scales. This is not an "
        "independent replay of the generator QC baseline."
    )
}

print("\nTRAIN-ONLY TRAJECTORY DESCRIPTIVES")
print(json.dumps(trajectory_summary, indent=2))

# ==========================================================================================
# 11 — INDEPENDENT LABEL RECONSTRUCTION
#
# Checks every saved window in TRAIN/VALIDATION/TEST.
#
# Does NOT fit models or evaluate prediction performance.
# ==========================================================================================

window_summaries = {}

all_keys = set()

for split in SPLITS:

    path = DATA / f"windows_W10_{split}.npz"

    print("\nVerifying:", path.name)

    with np.load(path, allow_pickle=False) as pack:

        required_arrays = [
            "X",
            "episode_id",
            "minute",
            "onset_class",
            "family_class",
            "y_horizons"
        ]

        absent = [
            a for a in required_arrays
            if a not in pack.files
        ]

        if absent:
            raise RuntimeError(
                f"{split} missing arrays: {absent}"
            )

        X = pack["X"]
        ids = pack["episode_id"].astype(str)
        t = pack["minute"].astype(int)

        y_onset = pack["onset_class"].astype(int)
        y_family = pack["family_class"].astype(int)
        y_h = pack["y_horizons"].astype(int)

        n = len(X)

        check(
            f"{split} W10 feature shape",
            X.ndim == 3
            and X.shape[1:] == (10, len(FEATURES)),
            f"Actual: {X.shape}"
        )

        check(
            f"{split} window row alignment",
            len(ids) == n
            and len(t) == n
            and len(y_onset) == n
            and len(y_family) == n
            and y_h.shape == (n, 3)
        )

        check(
            f"{split} finite window features",
            bool(np.isfinite(X).all())
        )

        check(
            f"{split} valid end-minute range",
            bool(
                ((t >= W - 1)
                & (t + MAX_H < EPISODE_MINUTES)).all()
            )
        )

        split_ids = set(
            meta.loc[
                meta.Split == split,
                "Episode_ID"
            ]
        )

        check(
            f"{split} episode ownership",
            set(ids).issubset(split_ids)
        )

        keys = pd.MultiIndex.from_arrays(
            [ids, t],
            names=["Episode_ID", "Minute"]
        )

        check(
            f"{split} unique window keys",
            bool(keys.is_unique)
        )

        cross_keys = set(zip(ids.tolist(), t.tolist()))

        check(
            f"{split} no cross-split sample duplication",
            all_keys.isdisjoint(cross_keys)
        )

        all_keys.update(cross_keys)

        episode_info = meta_lookup.loc[ids]

        onset = episode_info[
            "Onset_Minute"
        ].to_numpy(dtype=int)

        family = episode_info[
            "Family"
        ].astype(str).to_numpy()

        future = np.where(
            onset >= 0,
            onset - t,
            -1
        )

        expected_onset = np.where(
            (future >= 1) & (future <= MAX_H),
            future,
            0
        )

        family_mapping = {
            f: i + 1
            for i, f in enumerate(FAMILIES)
        }

        expected_family = np.array([
            family_mapping[f] if positive else 0
            for f, positive in zip(
                family,
                expected_onset > 0
            )
        ])

        expected_horizons = np.column_stack([
            (
                (expected_onset >= 1)
                &
                (expected_onset <= h)
            ).astype(int)
            for h in HORIZONS
        ])

        check(
            f"{split} exact onset target reconstruction",
            bool(
                np.array_equal(
                    y_onset,
                    expected_onset
                )
            )
        )

        check(
            f"{split} exact family target reconstruction",
            bool(
                np.array_equal(
                    y_family,
                    expected_family
                )
            )
        )

        check(
            f"{split} exact horizon target reconstruction",
            bool(
                np.array_equal(
                    y_h,
                    expected_horizons
                )
            )
        )

        check(
            f"{split} excludes at/post-onset windows",
            bool(
                ((onset < 0) | (t < onset)).all()
            )
        )

        # Joint competing-risk target:
        # 0 = no attack in 15 minutes
        # 1..75 = minute/family combination
        joint = np.where(
            expected_onset == 0,
            0,
            (expected_onset - 1) * len(FAMILIES)
            + expected_family
        )

        check(
            f"{split} 76-class joint target range",
            bool(
                ((joint >= 0) & (joint <= 75)).all()
            )
        )

        decoded_onset = np.where(
            joint == 0,
            0,
            ((joint - 1) // len(FAMILIES)) + 1
        )

        decoded_family = np.where(
            joint == 0,
            0,
            ((joint - 1) % len(FAMILIES)) + 1
        )

        check(
            f"{split} joint-target round-trip",
            bool(
                np.array_equal(
                    decoded_onset,
                    expected_onset
                )
                and
                np.array_equal(
                    decoded_family,
                    expected_family
                )
            )
        )

        # Independently rebuild every saved window from
        # the minute predictors file.
        minute_index = minutes.set_index(
            ["Episode_ID", "Minute"]
        )

        # Process in chunks to avoid large memory duplication.
        reconstruct_ok = True
        chunk_size = 2048

        for start_i in range(0, n, chunk_size):

            stop_i = min(n, start_i + chunk_size)

            chunk_ids = ids[start_i:stop_i]
            chunk_t = t[start_i:stop_i]

            expanded_ids = np.repeat(
                chunk_ids,
                W
            )

            expanded_minutes = (
                chunk_t[:, None]
                -
                np.arange(
                    W - 1,
                    -1,
                    -1
                )[None, :]
            ).reshape(-1)

            lookup_keys = pd.MultiIndex.from_arrays(
                [
                    expanded_ids,
                    expanded_minutes
                ],
                names=["Episode_ID", "Minute"]
            )

            rebuilt = minute_index.reindex(
                lookup_keys
            )[FEATURES].to_numpy(
                dtype=np.float32
            )

            rebuilt = rebuilt.reshape(
                -1,
                W,
                len(FEATURES)
            )

            if not np.allclose(
                rebuilt,
                X[start_i:stop_i],
                rtol=1e-6,
                atol=1e-5,
                equal_nan=False
            ):

                reconstruct_ok = False
                break

        check(
            f"{split} exact causal W10 reconstruction",
            reconstruct_ok
        )

        # Verify every eligible decision time exists once.
        expected_keys = set()

        for entry in meta.loc[
            meta.Split == split
        ].itertuples(index=False):

            stop = (
                min(
                    int(entry.Onset_Minute) - 1,
                    EPISODE_MINUTES - MAX_H - 1
                )
                if int(entry.Onset_Minute) >= 0
                else EPISODE_MINUTES - MAX_H - 1
            )

            for minute in range(W - 1, stop + 1):
                expected_keys.add(
                    (str(entry.Episode_ID), minute)
                )

        check(
            f"{split} exact eligible-window coverage",
            cross_keys == expected_keys,
            (
                f"Observed={len(cross_keys)}, "
                f"Expected={len(expected_keys)}"
            )
        )

        window_summaries[split] = {
            "rows": n,
            "shape": list(X.shape),
            "horizon_prevalence": (
                y_h.mean(axis=0).tolist()
            ),
            "joint_positive": int(
                (joint > 0).sum()
            ),
            "unique_episodes": int(
                len(set(ids))
            )
        }

# ==========================================================================================
# 12 — DATASET-LEVEL LIMITATIONS / NEGATIVE CONTROLS
#
# No diagnostic model is trained.
# The known shortcut results are NOT automatically treated
# as proof of full synthetic validity.
# ==========================================================================================

source_family = pd.crosstab(
    meta["Family"],
    meta["Source"]
)

single_source_families = []

for family in FAMILIES:

    available_sources = int(
        (source_family.loc[family] > 0).sum()
    )

    if available_sources == 1:
        single_source_families.append(family)

known_limitations = [
    "Pre-attack trajectories are synthetic assumptions, not observed real attack precursors.",
    "Calibration groups are reused within partitions, so episodes are not independent real traffic recordings.",
    "Matched H10 diagnostic changes prevalence and is not an operational performance estimate.",
    "Fixed simulated episode duration and a restricted onset interval remain design limitations.",
    "No empirical real-world forecast transfer has been demonstrated.",
    "The counterfactual unmodified baseline needed to reproduce original robust_signal QC is not saved.",
    "Pre-precursor predictive negative-control performance cannot be established without a separately specified diagnostic model."
]

if single_source_families:

    known_limitations.append(
        "Single-source attack families: "
        + ", ".join(single_source_families)
    )

print("\nSOURCE/FAMILY DISTRIBUTION")
print(source_family.to_string())

print("\nDOCUMENTED LIMITATIONS")

for item in known_limitations:
    print("-", item)

# ==========================================================================================
# 13 — SAVE REPORT OUTSIDE THE DATASET DIRECTORY
# ==========================================================================================

checks_df = pd.DataFrame(checks)

blockers = checks_df[
    (checks_df.Status == "FAIL")
    &
    (checks_df.Severity == "BLOCKER")
]

warnings = checks_df[
    (checks_df.Status == "FAIL")
    &
    (checks_df.Severity != "BLOCKER")
]

# PASS here means the checks implemented in this notebook passed.
# It does NOT certify real-world validity, family-prediction
# performance, or universal absence of generator shortcuts.
release_gate = (
    "TECHNICAL_CHECKS_PASS"
    if len(blockers) == 0
    else "TECHNICAL_CHECKS_FAIL"
)

report = {
    "dataset": "Synthetic_NAF_v2",
    "candidate": "release_candidate_corrected",
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "technical_gate": release_gate,
    "dataset_frozen": False,
    "model_training_performed": False,
    "test_predictions_evaluated": False,
    "total_checks": len(checks_df),
    "passed": int(
        (checks_df.Status == "PASS").sum()
    ),
    "failed": int(
        (checks_df.Status == "FAIL").sum()
    ),
    "blocking_failures": blockers[
        "Check"
    ].tolist(),
    "nonblocking_failures": warnings[
        "Check"
    ].tolist(),
    "windows": window_summaries,
    "train_trajectory_descriptives": trajectory_summary,
    "single_source_families": single_source_families,
    "known_limitations": known_limitations,
    "requires_manual_release_decision": True
}

checks_df.to_csv(
    REPORT_DIR / "FINAL_RELEASE_CHECKS.csv",
    index=False
)

trajectory.to_csv(
    REPORT_DIR / "EPISODE_TRAJECTORY_DESCRIPTIVES.csv",
    index=False
)

source_family.to_csv(
    REPORT_DIR / "SOURCE_FAMILY_DISTRIBUTION.csv"
)

save_json(
    REPORT_DIR / "FINAL_RELEASE_VERIFICATION_REPORT.json",
    report
)

# Hash only report artifacts here.
report_hashes = {}

for path in sorted(REPORT_DIR.iterdir()):

    if path.is_file() and path.name != "REPORT_SHA256SUMS.json":

        report_hashes[path.name] = sha256_file(path)

save_json(
    REPORT_DIR / "REPORT_SHA256SUMS.json",
    report_hashes
)

# ==========================================================================================
# 14 — FINAL CONSOLE REPORT
# ==========================================================================================

print("\n" + "=" * 105)
print("FINAL SYNTHETIC_NAF_V2 VERIFICATION REPORT")
print("=" * 105)

print("Technical gate:", release_gate)

print(
    "Checks passed:",
    int((checks_df.Status == "PASS").sum()),
    "/",
    len(checks_df)
)

print("Blocking failures:", len(blockers))
print("Nonblocking failures:", len(warnings))

print("\nForecasting windows:")

for split, info in window_summaries.items():

    print(
        split,
        info["shape"],
        "H5/H10/H15 prevalence:",
        np.round(
            info["horizon_prevalence"],
            4
        )
    )

print("\nSingle-source families:")
print(single_source_families)

print("\nFinal dataset decision: NOT AUTOMATICALLY FROZEN")
print("Model training: NOT PERFORMED")
print("Official TEST scoring: NOT PERFORMED")

print("\nReports saved to:")
print(REPORT_DIR)

if len(blockers) > 0:

    print("\nBLOCKING CHECK FAILURES:")
    print(
        blockers[
            ["Check", "Details"]
        ].to_string(index=False)
    )

else:

    print(
        "\nAll implemented technical checks passed."
    )
    print(
        "Scientific limitations and release decision "
        "still require review."
    )

print("\n" + "=" * 105)
print("VERIFICATION FINISHED — ORIGINAL DATASET UNCHANGED")
print("=" * 105)


In [ ]:

# =============================================================================
# SYNTHETIC_NAF_V2 — FORMAL DATASET RELEASE FREEZE
#
# Selected candidate:
#   Synthetic_NAF_v2/release_candidate_corrected
#
# PURPOSE:
#   - Verify approved technical QC results
#   - Verify all dataset file hashes
#   - Record feature, horizon, family and split contracts
#   - Record scientific limitations
#   - Create immutable-content reference manifests
#   - Declare formal release version 2.0.0
#
# DOES NOT:
#   - Modify dataset contents
#   - Regenerate any samples
#   - Train or evaluate any forecasting model
#   - Change labels or decision thresholds
#   - Unlock or evaluate TEST
#
# NOTE:
#   A manifest/hash freeze detects later modifications.
#   Google Drive files are not physically made read-only.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATA = ROOT / "Synthetic_NAF_v2" / "release_candidate_corrected"

VERIFICATION = ROOT / "Synthetic_NAF_v2" / "final_release_verification"

FREEZE = ROOT / "Synthetic_NAF_v2" / "FINAL_RELEASE_FREEZE"

VERSION = "2.0.0"
RELEASE_NAME = "Synthetic_NAF_v2"

REQUIRED_DATA_FILES = [
    "minute_predictors.parquet",
    "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv",
    "windows_W10_TRAIN.npz",
    "windows_W10_VALIDATION.npz",
    "windows_W10_TEST.npz",
    "FEATURE_AND_DATASET_CONTRACT.json",
    "QC_SUMMARY.json",
    "qc_rejections.csv",
    "SHA256SUMS.json",
]

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()

def canonical_json_bytes(value):
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path, value):
    path.write_text(
        json.dumps(value, indent=2, sort_keys=True),
        encoding="utf-8"
    )

print("=" * 100)
print("SYNTHETIC_NAF_V2 — FORMAL RELEASE FREEZE")
print("=" * 100)

# =============================================================================
# A. VERIFY INPUTS AND THE COMPLETED TECHNICAL GATE
# =============================================================================

for filename in REQUIRED_DATA_FILES:
    path = DATA / filename
    if not path.is_file():
        raise FileNotFoundError(path)

verification_path = (
    VERIFICATION / "FINAL_RELEASE_VERIFICATION_REPORT.json"
)
checks_path = VERIFICATION / "FINAL_RELEASE_CHECKS.csv"

if not verification_path.is_file():
    raise FileNotFoundError(verification_path)

if not checks_path.is_file():
    raise FileNotFoundError(checks_path)

verification = read_json(verification_path)
checks = pd.read_csv(checks_path)

if verification.get("technical_gate") != "TECHNICAL_CHECKS_PASS":
    raise RuntimeError("Technical verification gate did not pass.")

if verification.get("candidate") != "release_candidate_corrected":
    raise RuntimeError("Verification references a different candidate.")

if len(checks) != 100 or not (checks["Status"] == "PASS").all():
    raise RuntimeError(
        "Expected 100 successful technical verification checks."
    )

if int(verification.get("blocking_failures", -1).__len__()) != 0:
    raise RuntimeError("Blocking verification failures exist.")

if verification.get("model_training_performed") is not False:
    raise RuntimeError("Verification does not confirm no training.")

if verification.get("test_predictions_evaluated") is not False:
    raise RuntimeError("Verification does not confirm TEST embargo.")

print("✓ Technical verification: 100 / 100 PASS")
print("✓ Model training not performed")
print("✓ Official TEST not evaluated")

# =============================================================================
# B. REVERIFY DATASET HASHES
# =============================================================================

recorded_hashes = read_json(DATA / "SHA256SUMS.json")

actual_hashes = {}

for path in sorted(DATA.iterdir()):
    if path.is_file() and path.name != "SHA256SUMS.json":
        actual_hashes[path.name] = sha256_file(path)

if set(recorded_hashes) != set(actual_hashes):
    raise RuntimeError(
        "Dataset hash inventory does not match actual files."
    )

for filename, actual in actual_hashes.items():
    if recorded_hashes[filename] != actual:
        raise RuntimeError(
            f"Dataset file changed: {filename}"
        )

manifest_file_hashes = {
    **actual_hashes,
    "SHA256SUMS.json": sha256_file(DATA / "SHA256SUMS.json")
}

print("✓ Dataset hashes verified")
print("✓ All release-candidate files accounted for")

# =============================================================================
# C. VERIFY CORE CONTRACT
# =============================================================================

contract = read_json(
    DATA / "FEATURE_AND_DATASET_CONTRACT.json"
)

features = contract["features"]
families = contract["attack_families"]

assert len(features) == 14
assert len(families) == 5

assert contract["history_minutes"] == 10
assert contract["forecast_horizons"] == [5, 10, 15]

assert contract["episode_count"] == 2000
assert contract["attack_episodes"] == 1500
assert contract["benign_episodes"] == 500

expected_splits = {
    "TRAIN": 1400,
    "VALIDATION": 300,
    "TEST": 300,
}

meta = pd.read_csv(
    DATA / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
)

actual_splits = meta.groupby("Split").size().to_dict()

if actual_splits != expected_splits:
    raise RuntimeError(
        f"Split mismatch: {actual_splits}"
    )

print("✓ Core feature and forecasting contracts verified")

# =============================================================================
# D. RELEASE MANIFEST
# =============================================================================

limitations = [
    "Attack precursors are simulated rather than observed in real network traces.",
    "Real-world forecasting performance is not established.",
    "Calibration groups are reused within partitions.",
    "BRUTE_AUTH is calibrated only from CIC_IDS2017.",
    "FUZZING is calibrated only from UNSW_NB15.",
    "Episode duration and attack onset belong to a controlled simulation.",
    "The matched H10 diagnostic does not represent operational class prevalence.",
    "The original unmodified counterfactual baseline is not retained.",
    "Further external validation is required before deployment claims."
]

release_manifest = {
    "release_name": RELEASE_NAME,
    "release_version": VERSION,
    "release_status": "FROZEN_CONTROLLED_SYNTHETIC_BENCHMARK",
    "release_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_folder": str(DATA),
    "dataset_kind": "SOURCE_CALIBRATED_SYNTHETIC_FORECASTING",
    "episodes": {
        "total": 2000,
        "attack": 1500,
        "benign": 500,
        "hard_benign": 250,
        "split_counts": expected_splits
    },
    "temporal_contract": {
        "cadence_minutes": 1,
        "history_minutes": 10,
        "forecast_horizons_minutes": [5, 10, 15],
        "attack_onset_minute_range": [46, 82],
        "attack_precursor_minutes_range": [15, 30]
    },
    "feature_contract": {
        "predictor_count": len(features),
        "predictors": features,
        "attack_families": families,
        "metadata_as_predictor": False,
        "timestamps_as_predictor": False,
        "future_labels_as_predictor": False
    },
    "forecasting_targets": {
        "onset_class": "0=no onset within H15, 1..15=minutes to onset",
        "family_class": "0=no event within H15, 1..5=family index",
        "joint_competing_risk_classes": 76,
        "horizons": ["H5", "H10", "H15"]
    },
    "data_integrity": {
        "verification_gate": verification["technical_gate"],
        "verification_checks_passed": 100,
        "file_sha256": manifest_file_hashes
    },
    "test_policy": {
        "status": "EMBARGOED_FOR_MODEL_EVALUATION",
        "final_evaluation_runs_allowed": 1,
        "training_allowed_on_test": False,
        "hyperparameter_optimization_on_test": False
    },
    "limitations": limitations,
    "model_training_started": False,
    "real_world_validity_claimed": False,
    "allowed_post_freeze_changes": [
        "Documented implementation-bug correction in a new dataset version"
    ],
    "disallowed_post_freeze_changes": [
        "Changing generator parameters to improve model scores",
        "Modifying labels based on validation performance",
        "Changing the TEST partition",
        "Adding features because a model performs poorly"
    ]
}

# =============================================================================
# E. WRITE FREEZE RECORD — NEVER OVERWRITE AN EXISTING RELEASE
# =============================================================================

if FREEZE.exists() and any(FREEZE.iterdir()):
    raise RuntimeError(
        f"Freeze folder already contains files: {FREEZE}. "
        "An existing release must not be overwritten."
    )

FREEZE.mkdir(parents=True, exist_ok=True)

manifest_bytes = canonical_json_bytes(release_manifest)
manifest_hash = sha256_bytes(manifest_bytes)

write_json(
    FREEZE / "Synthetic_NAF_v2_RELEASE_MANIFEST.json",
    release_manifest
)

lock_record = {
    "release": RELEASE_NAME,
    "version": VERSION,
    "manifest_sha256_canonical": manifest_hash,
    "dataset_folder": str(DATA),
    "dataset_hashes_verified": True,
    "technical_checks_passed": 100,
    "status": "LOGICALLY_FROZEN",
    "note": (
        "The release is logically frozen by immutable reference "
        "hashes. Google Drive permissions and physical file "
        "write-protection are not changed."
    )
}

write_json(
    FREEZE / "Synthetic_NAF_v2_FREEZE_LOCK.json",
    lock_record
)

readme = """# Synthetic_NAF_v2 — Frozen Controlled Synthetic Benchmark

Version: 2.0.0

Dataset:
Synthetic_NAF_v2/release_candidate_corrected

Technical verification:
100/100 checks passed.

This release is used for research into forecasting
simulated network attack onset from prior network behaviour.

It supports:
- W10 causal forecasting
- H5, H10 and H15 onset horizons
- Five synthetic attack families
- Discrete onset/family joint targets
- Episode-level TRAIN/VALIDATION/TEST separation

TEST is embargoed for final model evaluation.

This dataset does not establish real-world pre-attack
forecasting reliability.

After freezing, performance-based regeneration is forbidden.

Any genuine implementation bug requires:
- a documented correction
- a new version
- complete model retraining and reevaluation

No model training was performed during freezing.
"""

(FREEZE / "README_RELEASE.md").write_text(
    readme,
    encoding="utf-8"
)

freeze_hashes = {
    path.name: sha256_file(path)
    for path in sorted(FREEZE.iterdir())
    if path.is_file()
}

write_json(
    FREEZE / "FREEZE_ARTIFACT_SHA256SUMS.json",
    freeze_hashes
)

# =============================================================================
# F. FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)
print("SYNTHETIC_NAF_V2 — RELEASE FREEZE COMPLETE")
print("=" * 100)

print("Release:", RELEASE_NAME)
print("Version:", VERSION)
print("Status: FROZEN CONTROLLED SYNTHETIC BENCHMARK")
print("Technical checks: 100 / 100 PASS")
print("Dataset source:", DATA)
print("Release records:", FREEZE)
print("Manifest SHA256:", manifest_hash)

print("\nDataset regeneration: NOT PERFORMED")
print("Model training: NOT PERFORMED")
print("TEST evaluation: NOT PERFORMED")
print("Original dataset files: UNCHANGED")

print("\nNEXT: Review and explicitly authorize the model-training protocol.")
